# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and sets working directory to the repo root. Locally it locates the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Sujan-lab-cell/flyrank-ml-internship'
REPO_DIR = 'flyrank-ml-internship'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir('data/raw') and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir('..')

print('Working dir:', os.getcwd())
assert os.path.exists('data/raw/content_refresh_anonymized.csv'), 'starter CSV not found — are you at the repo root?'
print('Starter data found. You are ready.')

Working dir: e:\FlyRank_Internship\flyrank-ml-internship
Starter data found. You are ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

I chose **Lane 2: Refresh / Content Opportunity Scoring** because it focuses on using supervised machine learning to prioritize existing webpages for updates, expansion, or performance protection. In search engine optimization (SEO), content teams waste significant resources manually auditing pages or guessing which declining content to refresh. By framing this as an ML scoring problem, we can identify high-value content with dropping organic performance and prioritize editor time where refresh action yields the highest expected ROI.

In [2]:
# Code check: Inspect dataset size and count of content items across clients
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total Content Items (Rows): {len(df):,}")
print(f"Total Unique Clients: {df['client_id'].nunique()}")
print("\nContent Types Breakdown:")
print(df['content_type'].value_counts())

Total Content Items (Rows): 30,000
Total Unique Clients: 32

Content Types Breakdown:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

* **Decision Improved:** Which specific content URLs should be scheduled for content refresh or optimization in the upcoming week/month, ranked by priority.
* **Who Acts on It:** Content strategists, SEO managers, and editorial teams.
* **Action Taken:** Conducting content audits, rewriting outdated sections, optimizing target keywords, or adding internal links to declining pages.
* **Cost of a Wrong Call:**
  * *False Positive (recommending a page that doesn't need refresh):* Wasted editorial hours and budget on healthy pages that wouldn't benefit from updates.
  * *False Negative (missing a severely declining high-traffic page):* Continued loss of organic rankings, traffic, and revenue to competing search results.
* **Why ML helps over a simple rule:** A simple threshold (e.g. `days_since_last_update > 180`) fails because many old pages maintain strong position/CTR, while newer pages can rapidly drop off due to intent shifts or competition. ML combines non-linear signals (search volume, CPC, impressions trend, engagement, position tier) to rank true opportunity.

In [3]:
# Code check: Quantify the opportunity and risk in traffic terms
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Distribution of trend directions
trend_counts = df['trend_direction'].value_counts()
print("Trend Direction Breakdown:")
print(trend_counts)

# Quantify total 90-day impressions for declining pages
declining_df = df[df['trend_direction'] == 'down']
print(f"\nDeclining Pages Count: {len(declining_df):,} ({len(declining_df)/len(df):.1%})")
print(f"Impressions at Risk (Declining Pages): {declining_df['impressions_90d'].sum():,.0f}")

Trend Direction Breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining Pages Count: 16,262 (54.2%)
Impressions at Risk (Declining Pages): 79,994,363


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

1. **High Rate of Performance Decline:** Out of **30,000** content pages across 32 clients, **16,262 pages (54.2%)** exhibit a downward performance trend (`trend_direction == 'down'`).
2. **Huge Traffic Exposure at Risk:** Declining pages account for **79,994,363 organic impressions** out of 156M total 90-day impressions (**51.3%** of all search visibility across the dataset).
3. **Striking Distance Opportunity:** **4,452** declining pages are currently in the **'striking' position tier** (average position 11–20). These represent high-leverage refresh targets where a minor boost can move pages to Page 1 of search results.

In [4]:
# Code check: Calculate exact 3 real numbers supporting Lane 2
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_rows = len(df)
n_down = (df['trend_direction'] == 'down').sum()
pct_down = (n_down / total_rows) * 100

total_impressions = df['impressions_90d'].sum()
down_impressions = df[df['trend_direction'] == 'down']['impressions_90d'].sum()
pct_down_impressions = (down_impressions / total_impressions) * 100

striking_down = df[(df['trend_direction'] == 'down') & (df['position_tier'] == 'striking')].shape[0]
striking_total = df[df['position_tier'] == 'striking'].shape[0]

print("=== REAL NUMBERS FROM STARTER DATASET ===")
print(f"1. Total Pages: {total_rows:,} | Declining Pages: {n_down:,} ({pct_down:.1f}%)")
print(f"2. Total 90d Impressions: {total_impressions:,.0f} | Impressions at Risk: {down_impressions:,.0f} ({pct_down_impressions:.1f}%)")
print(f"3. Striking Distance Pages Declining: {striking_down:,} out of {striking_total:,} ({striking_down/striking_total:.1%})")

=== REAL NUMBERS FROM STARTER DATASET ===
1. Total Pages: 30,000 | Declining Pages: 16,262 (54.2%)
2. Total 90d Impressions: 156,010,989 | Impressions at Risk: 79,994,363 (51.3%)
3. Striking Distance Pages Declining: 4,452 out of 7,304 (61.0%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN claim:**
* **Observed Correlations:** Documented historical patterns where specific combinations of features (e.g. content age, CTR, position tier, engagement rate) correlate with performance drop-off.
* **Directional Priority:** A decision-support ranking queue that prioritizes pages with higher predicted likelihood of decline and higher impression impact.
* **Measured Baseline Performance:** Quantifiable precision@K and ROC-AUC metrics demonstrating that the model prioritizes declining content better than random choice or a single heuristic rule.

**What this work CANNOT claim:**
* **Causal Proof:** We cannot claim that updating content *causes* a ranking recovery or that age *caused* a decline (correlation does not equal causation).
* **Predicting Google's Algorithm:** We do not reverse-engineer or predict Google's proprietary search ranking algorithms.
* **Guaranteed Ranking Outcomes:** We cannot promise ranking positions or traffic numbers after a refresh action is taken.

In [5]:
# Code check: Verify target label distribution and feature integrity (prevent leakage)
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Confirm target label distribution and verify trend_pct derivation
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Label Distribution (is_declining_label):")
print(df['is_declining_label'].value_counts(normalize=True))

# Verification: Confirm trend_pct and trend_direction are target derivations, not features
print("\nTarget leakage guard check:")
print("Features must exclude: ['trend_direction', 'trend_pct', 'is_declining_label']")

Label Distribution (is_declining_label):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64

Target leakage guard check:
Features must exclude: ['trend_direction', 'trend_pct', 'is_declining_label']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.